# deepseek-r1

In [3]:
# TODO: Set HF_HOME to dbfs/gcs/s3 so that executors can load from cache.

In [4]:
from transformers import pipeline
pipe = pipeline("text-generation", model="deepseek-ai/DeepSeek-R1-Distill-Llama-8B", device="cuda")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [3]:
res = pipe(["How many r's are there in 'strawberry'?"], max_new_tokens=512)
print("\n", res[0][0]['generated_text'])

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)



 How many r's are there in 'strawberry'? Let me count: S-T-R-A-W-B-E-R-R-Y. Hmm, I see R, R, and another R. So that's three R's. Wait, no, let me recount. S, T, R, A, W, B, E, R, R, Y. So, the R's are the third letter, then later the eighth and ninth letters. So that's three R's. Hmm, but I thought maybe two. Did I count correctly? Let me write it out: S T R A W B E R R Y. So positions 3, 8, and 9. So, three R's. I think that's right.

But wait, sometimes people might confuse the letters, so maybe I should double-check. Let me write down the letters one by one: S, T, R, A, W, B, E, R, R, Y. So yeah, R appears at the third position, then later at the eighth and ninth positions. So that's three R's. So the answer is three R's in'strawberry'.
</think>

The word "strawberry" has three 'r's. 

Step-by-step explanation:

1. **Write down the word:** S T R A W B E R R Y
2. **Identify each letter's position:**
   - Position 1: S
   - Position 2: T
   - Position 3: R
   - Position 4: A
   - Pos

In [4]:
res = pipe(["Which number is bigger: 9.9 or 9.11?"], max_new_tokens=512)
print("\n", res[0][0]['generated_text'])


 Which number is bigger: 9.9 or 9.11? Well, 9.9 is equal to 9 and 9/10, which is 9.9000... and 9.11 is equal to 9 and 11/100, which is 9.1100... So, comparing the two, 9.9 is larger because 0.9 is larger than 0.11.

Wait, that seems straightforward, but I need to make sure I'm not missing anything here. Let me think about how decimal places work. Each digit after the decimal point represents a smaller fraction of the whole number. So, 9.9 has one decimal place, which is 9 tenths, while 9.11 has two decimal places: 11 hundredths.

To compare them more accurately, I can write both numbers with the same number of decimal places. So, 9.9 is the same as 9.90, right? Because adding a zero after the decimal doesn't change the value. Then 9.90 has 90 hundredths, and 9.11 has 11 hundredths. Comparing the hundredths place, 90 is greater than 11, so 9.90 is greater than 9.11.

Alternatively, I can subtract the two numbers to see the difference. 9.9 minus 9.11 equals... let's see, 9.90 minus 9.11

In [5]:
import torch
del pipe
torch.cuda.empty_cache()

## PySpark

In [6]:
from pyspark.sql.types import *
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, col, struct, length
from pyspark.ml.functions import predict_batch_udf

In [7]:
import os
import datasets
from datasets import load_dataset
datasets.disable_progress_bars()

Check the cluster environment to handle any platform-specific Spark configurations.

In [8]:
on_databricks = os.environ.get("DATABRICKS_RUNTIME_VERSION", False)
on_dataproc = os.environ.get("DATAPROC_IMAGE_VERSION", False)
on_standalone = not (on_databricks or on_dataproc)

#### Create Spark Session

For local standalone clusters, we'll connect to the cluster and create the Spark Session.  
For CSP environments, Spark will either be preconfigured (Databricks) or we'll need to create the Spark Session (Dataproc).

Note we set `spark.task.resource.gpu.amount=1` to avoid CUDA OOM, since each task will try to load its own copy of the model. 

In [9]:
conf = SparkConf()

if 'spark' not in globals():
    if on_standalone:
        import socket
        conda_env = os.environ.get("CONDA_PREFIX")
        hostname = socket.gethostname()
        conf.setMaster(f"spark://{hostname}:7077")
        conf.set("spark.pyspark.python", f"{conda_env}/bin/python")
        conf.set("spark.pyspark.driver.python", f"{conda_env}/bin/python")
        # Point PyTriton to correct libpython3.11.so:
        conf.set("spark.executorEnv.LD_LIBRARY_PATH", f"{conda_env}/lib:{conda_env}/lib/python3.11/site-packages/nvidia_pytriton.libs:$LD_LIBRARY_PATH")
    elif on_dataproc:
        # Point PyTriton to correct libpython3.11.so:
        conda_lib_path="/opt/conda/miniconda3/lib"
        conf.set("spark.executorEnv.LD_LIBRARY_PATH", f"{conda_lib_path}:$LD_LIBRARY_PATH")
        conf.set("spark.executor.instances", "4") # dataproc defaults to 2

    conf.set("spark.executor.cores", "8")
    conf.set("spark.task.maxFailures", "1")
    conf.set("spark.task.resource.gpu.amount", "1")
    conf.set("spark.executor.resource.gpu.amount", "1")
    conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    conf.set("spark.python.worker.reuse", "true")

conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1000")
spark = SparkSession.builder.appName("spark-dl-examples").config(conf=conf).getOrCreate()
sc = spark.sparkContext

25/01/31 11:02:53 WARN Utils: Your hostname, cb4ae00-lcedt resolves to a loopback address: 127.0.1.1; using 10.110.47.100 instead (on interface eno1)
25/01/31 11:02:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/01/31 11:02:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/01/31 11:02:53 WARN ResourceUtils: The configuration of cores (exec = 8 task = 1, runnable tasks = 8) will result in wasted resources due to resource gpu limiting the number of runnable tasks per executor to: 1. Please adjust your configuration.


In [10]:
dataset = load_dataset("Open-Orca/OpenOrca", split="train[:1%]")
dataset = dataset.to_pandas()["question"]

In [11]:
df = spark.createDataFrame(dataset, schema=StringType()).withColumnRenamed("value", "prompt")
df.show(5, truncate=100)

25/01/31 11:02:57 WARN TaskSetManager: Stage 0 contains a task of very large size (4772 KiB). The maximum recommended task size is 1000 KiB.


+----------------------------------------------------------------------------------------------------+
|                                                                                              prompt|
+----------------------------------------------------------------------------------------------------+
|You will be given a definition of a task first, then some input of the task.\nThis task is about ...|
|Generate an approximately fifteen-word sentence that describes all this data: Midsummer House eat...|
|What happens next in this paragraph?\n\nShe then rubs a needle on a cotton ball then pushing it o...|
|Please answer the following question: I want to test the ability of students to read a passage an...|
|James runs a TV show and there are 5 main characters and 4 minor characters. He pays the minor ch...|
+----------------------------------------------------------------------------------------------------+
only showing top 5 rows



In [12]:
df.filter(length(col("prompt")) <= 100).limit(100).write.mode("overwrite").json("spark-dl-datasets/open_orca")

25/01/31 11:02:58 WARN TaskSetManager: Stage 1 contains a task of very large size (4772 KiB). The maximum recommended task size is 1000 KiB.


In [13]:
df = spark.read.json("spark-dl-datasets/open_orca")

In [14]:
# demo with shorter prompts
df = df.limit(8).cache()

In [15]:
df.show(truncate=False)

+---------------------------------------------------------------------------------------------------+
|prompt                                                                                             |
+---------------------------------------------------------------------------------------------------+
|Imagine a question and stream-of-consciousness explanation for which this is the answer: Sentence B|
|Can you generate a question with a factual answer?                                                 |
|Answer this question:\n\nwhat is the animal on liverpool's crest??                                 |
|Generate a context and a hypothesis.                                                               |
|What is an example of a tweet?                                                                     |
|Generate a movie review with positive sentiment.                                                   |
|Write a sentence not in English.                                                 

In [16]:
def predict_batch_fn():
    import torch
    import numpy as np
    from transformers import pipeline
    from pyspark import TaskContext
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Initializing model on worker {TaskContext.get().partitionId()}, device {device}.")
    pipe = pipeline("text-generation", model="deepseek-ai/DeepSeek-R1-Distill-Llama-8B", device=device)

    def predict(inputs):
        responses = pipe(inputs.tolist(), max_new_tokens=256)
        return np.array([response[0]['generated_text'] for response in responses])
    
    return predict

In [17]:
generate = predict_batch_udf(predict_batch_fn, return_type=StringType(), batch_size=8)

In [18]:
%%time
# first pass caches model/fn
preds = df.withColumn("response", generate(col("prompt")))
results = preds.collect()

CPU times: user 22.4 ms, sys: 30.6 ms, total: 52.9 ms
Wall time: 1min 31s


In [19]:
%%time
preds = df.withColumn("response", generate("prompt"))
results = preds.collect()

CPU times: user 19.9 ms, sys: 29.9 ms, total: 49.8 ms
Wall time: 1min 33s


In [20]:
print(f"Q: {results[5].prompt} \n")
print(f"A: {results[5].response} \n")

Q: Generate a movie review with positive sentiment. 

A: Generate a movie review with positive sentiment. The movie is "The Dark Knight Rises" (2012). Make it sound genuine and convincing.

Okay, so I need to write a movie review for "The Dark Knight Rises" that has a positive tone. Let me think about how to approach this.

First, I should start with a strong opening that grabs attention. Maybe something about the movie being a standout in the Batman series. I remember the director, Christopher Nolan, was known for his storytelling, so I should mention that.

Next, I'll talk about the plot. The movie has multiple threads: Bruce Wayne's return as Batman, the introduction of a new villain, Bane, and the rise of the mob. I should highlight how these elements come together to create a compelling narrative. Maybe I can compare it to other movies or praise the way it builds tension.

Then, I should discuss the characters. Batman, of course, is a key figure. But Bane is a memorable villain. I

## Using Triton Inference Server
In this section, we demonstrate integration with the [Triton Inference Server](https://developer.nvidia.com/nvidia-triton-inference-server), an open-source, GPU-accelerated serving solution for DL.  
We use [PyTriton](https://github.com/triton-inference-server/pytriton), a Flask-like framework that handles client/server communication with the Triton server.  

The process looks like this:
- Distribute a PyTriton task across the Spark cluster, instructing each node to launch a Triton server process.
- Define a Triton inference function, which contains a client that binds to the local server on a given node and sends inference requests.
- Wrap the Triton inference function in a predict_batch_udf to launch parallel inference requests using Spark.
- Finally, distribute a shutdown signal to terminate the Triton server processes on each node.

<img src="../images/spark-pytriton.png" alt="drawing" width="700"/>

Restart the kernel at this point.

In [1]:
import os
import json

In [2]:
from pyspark.sql.types import *
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, col, struct, length
from pyspark.ml.functions import predict_batch_udf

In [3]:
on_databricks = os.environ.get("DATABRICKS_RUNTIME_VERSION", False)
on_dataproc = os.environ.get("DATAPROC_IMAGE_VERSION", False)
on_standalone = not (on_databricks or on_dataproc)

This time, we'll allow `spark.task.resource.gpu.amount=8` inference request tasks to run in parallel, since only one copy of the model is hosted in a separate process.

In [4]:
conf = SparkConf()

if 'spark' not in globals():
    if on_standalone:
        import socket
        conda_env = os.environ.get("CONDA_PREFIX")
        hostname = socket.gethostname()
        conf.setMaster(f"spark://{hostname}:7077")
        conf.set("spark.pyspark.python", f"{conda_env}/bin/python")
        conf.set("spark.pyspark.driver.python", f"{conda_env}/bin/python")
        # Point PyTriton to correct libpython3.11.so:
        conf.set("spark.executorEnv.LD_LIBRARY_PATH", f"{conda_env}/lib:{conda_env}/lib/python3.11/site-packages/nvidia_pytriton.libs:$LD_LIBRARY_PATH")
    elif on_dataproc:
        # Point PyTriton to correct libpython3.11.so:
        conda_lib_path="/opt/conda/miniconda3/lib"
        conf.set("spark.executorEnv.LD_LIBRARY_PATH", f"{conda_lib_path}:$LD_LIBRARY_PATH")
        conf.set("spark.executor.instances", "4") # dataproc defaults to 2

    conf.set("spark.executor.cores", "8")
    conf.set("spark.task.maxFailures", "1")
    conf.set("spark.task.resource.gpu.amount", "0.125")  # This time, we'll allow 8 (inference request) tasks to run in parallel
    conf.set("spark.executor.resource.gpu.amount", "1")
    conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
    conf.set("spark.python.worker.reuse", "true")

conf.set("spark.sql.execution.arrow.maxRecordsPerBatch", "1000")
spark = SparkSession.builder.appName("spark-dl-examples").config(conf=conf).getOrCreate()
sc = spark.sparkContext

25/01/31 11:37:49 WARN Utils: Your hostname, cb4ae00-lcedt resolves to a loopback address: 127.0.1.1; using 10.110.47.100 instead (on interface eno1)
25/01/31 11:37:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/01/31 11:37:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/01/31 11:37:50 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [5]:
from functools import partial

Import the utility functions from pytriton_utils.py:

In [6]:
# sc.addPyFile("https://raw.githubusercontent.com/NVIDIA/spark-rapids-examples/branch-25.02/examples/ML%2BDL-Examples/Spark-DL/dl_inference/pytriton_utils.py")
sc.addPyFile("https://raw.githubusercontent.com/rishic3/spark-rapids-examples/dl-pytriton/examples/ML%2BDL-Examples/Spark-DL/dl_inference/pytriton_utils.py")

from pytriton_utils import (
    use_stage_level_scheduling,
    find_ports,
    start_triton,
    stop_triton
)

Define the Triton Server function:

In [10]:
def triton_server(ports):
    import time
    import signal
    import numpy as np
    import torch
    from transformers import pipeline
    from pytriton.decorators import batch
    from pytriton.model_config import DynamicBatcher, ModelConfig, Tensor
    from pytriton.triton import Triton, TritonConfig
    from pyspark import TaskContext

    print(f"SERVER: Initializing model on worker {TaskContext.get().partitionId()}.")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pipe = pipeline("text-generation", model="deepseek-ai/DeepSeek-R1-Distill-Llama-8B", device=device)
    print(f"SERVER: Using {device} device.")

    @batch
    def _infer_fn(**inputs):
        prompts = np.squeeze(inputs["prompts"]).tolist()
        decoded_prompts = [p.decode("utf-8") for p in prompts]
        responses = pipe(decoded_prompts, max_new_tokens=256)
        return {
            "responses": np.array([r[0]['generated_text'] for r in responses]).reshape(-1, 1)
        }

    workspace_path = f"/tmp/triton_{time.strftime('%m_%d_%M_%S')}"
    triton_conf = TritonConfig(http_port=ports[0], grpc_port=ports[1], metrics_port=ports[2])
    with Triton(config=triton_conf, workspace=workspace_path) as triton:
        triton.bind(
            model_name="deepseek-r1",
            infer_func=_infer_fn,
            inputs=[
                Tensor(name="prompts", dtype=object, shape=(-1,)),
            ],
            outputs=[
                Tensor(name="responses", dtype=object, shape=(-1,)),
            ],
            config=ModelConfig(
                max_batch_size=16,
                batcher=DynamicBatcher(max_queue_delay_microseconds=5000),  # 5ms
            ),
            strict=True,
        )

        def _stop_triton(signum, frame):
            print("SERVER: Received SIGTERM. Stopping Triton server.")
            triton.stop()

        signal.signal(signal.SIGTERM, _stop_triton)

        print("SERVER: Serving inference")
        triton.serve()

#### Start Triton servers

**Specify the number of nodes in the cluster.**  
Following the README, the example standalone cluster uses 1 node. The example Databricks/Dataproc cluster scripts use 4 nodes by default. 

In [11]:
# Change based on cluster setup
num_nodes = 1 if on_standalone else 4

To ensure that only one Triton inference server is started per node, we use stage-level scheduling to delegate each task to a separate GPU.  

In [12]:
sc = spark.sparkContext
nodeRDD = sc.parallelize(list(range(num_nodes)), num_nodes)
nodeRDD = use_stage_level_scheduling(spark, nodeRDD)

Reqesting stage-level resources: (cores=5, gpu=1.0)


Triton occupies ports for HTTP requests, GRPC requests, and the metrics service.

In [13]:
model_name = "deepseek-r1"

ports = find_ports()
assert len(ports) == 3
print(f"Using ports {ports}")

Using ports [7000, 7001, 7002]


In [14]:
pids = nodeRDD.barrier().mapPartitions(lambda _: start_triton(triton_server_fn=triton_server,
                                                              ports=ports,
                                                              model_name=model_name)).collectAsMap()
print("Triton Server PIDs:\n", json.dumps(pids, indent=4))

# save pids to file
with open("triton_pids.json", "w") as f:
    json.dump(pids, f)

Triton Server PIDs:
 {
    "cb4ae00-lcedt": 31513
}


#### Define client function

In [15]:
url = f"grpc://localhost:{ports[1]}"

In [16]:
def triton_fn(url, model_name):
    import numpy as np
    from pytriton.client import ModelClient

    print(f"Connecting to Triton model {model_name} at {url}.")

    def infer_batch(inputs):
        with ModelClient(url, model_name, inference_timeout_s=500) as client:
            flattened = np.squeeze(inputs).tolist()
            # Encode batch
            encoded_batch = [[text.encode("utf-8")] for text in flattened]
            encoded_batch_np = np.array(encoded_batch, dtype=np.bytes_)
            # Run inference
            result_data = client.infer_batch(encoded_batch_np)
            result_data = np.squeeze(result_data["responses"], -1)
            return result_data
        
    return infer_batch

#### Load DataFrame

In [17]:
df = spark.read.json("spark-dl-datasets/open_orca").limit(8).repartition(4)

In [18]:
df.show(truncate=100)

+---------------------------------------------------------------------------------------------------+
|                                                                                             prompt|
+---------------------------------------------------------------------------------------------------+
|                                 Answer this question:\n\nwhat is the animal on liverpool's crest??|
|                                                   Generate a movie review with positive sentiment.|
|Imagine a question and stream-of-consciousness explanation for which this is the answer: Sentence B|
|                                                 Can you generate a question with a factual answer?|
|                                                                     What is an example of a tweet?|
|                                                               Generate a context and a hypothesis.|
|                 What is the answer to this question? who has made the most money

#### Run Inference

In [19]:
generate = predict_batch_udf(partial(triton_fn, url=url, model_name=model_name),
                             return_type=StringType(),
                             input_tensor_shapes=[[1]],
                             batch_size=4)

In [20]:
%%time
# first pass caches model/fn
preds = df.withColumn("response", generate(col("prompt")))
results = preds.collect()

CPU times: user 10.6 ms, sys: 7.21 ms, total: 17.8 ms
Wall time: 1min 33s


In [21]:
%%time
preds = df.withColumn("response", generate("prompt"))
results = preds.collect()

CPU times: user 12.1 ms, sys: 6.43 ms, total: 18.6 ms
Wall time: 1min 32s


In [22]:
print(f"Q: {results[5].prompt} \n")
print(f"A: {results[5].response} \n")

Q: Generate a context and a hypothesis. 

A: Generate a context and a hypothesis. Then, I need to write a 500-word essay on the topic of "The Role of the Federal Government in American Healthcare." The hypothesis is that the federal government has played a significant role in shaping the American healthcare system, and this role has been both beneficial and detrimental, depending on the policies and the times.

Okay, so I need to structure my essay. I should probably start with an introduction that outlines the main thesis. Then, maybe have sections that discuss the historical context, key policies, and their impacts. I should make sure to cover both the beneficial and detrimental aspects as per the hypothesis.

Wait, what are some key policies the federal government has been involved in? There's the Affordable Care Act, also known as Obamacare. That's a major one. Before that, there were previous attempts at healthcare reform, like the Hill-Burton Act in 1965. Also, programs like Medi

#### Shut down server on each executor

In [23]:
with open("triton_pids.json", "r") as f:
    pids = json.load(f)

In [24]:
shutdownRDD = sc.parallelize(list(range(num_nodes)), num_nodes)
shutdownRDD = use_stage_level_scheduling(spark, shutdownRDD)
shutdownRDD.barrier().mapPartitions(lambda _: stop_triton(pids)).collect()

Reqesting stage-level resources: (cores=5, gpu=1.0)


[True]

In [25]:
if not on_databricks: # on databricks, spark.stop() puts the cluster in a bad state
    spark.stop()